# SymPyによる数値解析：差分公式の記号的導出と誤差解析

## 概要
数値流体力学（CFD）や構造解析などの分野では、微分方程式を数値的に解くために「有限差分法」が広く用いられる。この手法では、微分を差分商で近似するが、高精度なスキーム（近似式）を導出するには、テイラー展開を用いた煩雑な係数決定作業が必要となる。

Pythonの数式処理ライブラリ **SymPy** を用いれば、この「数値計算のための数式変形」を自動化できる。任意の次数、任意のステンシル（参照点）に対する差分公式を導出し、その打ち切り誤差（Truncation Error）まで記号的に評価することが可能である。

本記事では、SymPyを用いて以下のトピックを解説する。

1. **テイラー展開の記号処理**：関数の局所的な振る舞いの記述
2. **基本的な差分公式**：前進・後退・中心差分の導出と誤差解析
3. **高次スキームの自動導出**：連立方程式を用いたステンシル係数の決定

### 筆者の環境
筆者の実行環境は以下の通りである。

In [ ]:
!sw_vers

In [ ]:
!python -V

必要なライブラリをインポートする。`Function` や `series` が中心的な役割を果たす。

In [ ]:
import sympy
from sympy import symbols, Function, series, factorial, solve, Matrix, simplify, Rational
from pprint import pprint as py_pprint

# 数式をLaTeX形式で綺麗に表示するための設定
sympy.init_printing()

print("sympy version :", sympy.__version__)

## 1. テイラー展開の記号処理

関数 $f(x)$ の $x+h$ における値は、テイラー展開によって以下のように記述される。

$$ f(x+h) = f(x) + h f'(x) + \frac{h^2}{2!} f''(x) + \frac{h^3}{3!} f'''(x) + \dots $$

SymPyでこれを表現してみる。

In [ ]:
x, h = symbols('x h', real=True)
f = Function('f')(x)

# f(x+h) を h について 4次まで展開 (O(h^4))
f_xh = f.subs(x, x+h).series(h, 0, 4)

print("f(x+h) のテイラー展開:")
sympy.pprint(f_xh)

## 2. 基本的な差分公式の導出

1階微分 $f'(x)$ を近似する最も基本的な「中心差分公式」を導出する。

$$ \frac{f(x+h) - f(x-h)}{2h} \approx f'(x) $$

この近似の精度（誤差のオーダー）を確認する。

In [ ]:
# f(x-h) の展開
f_xh_minus = f.subs(x, x-h).series(h, 0, 4)

# 中心差分の式を構成
central_diff = (f_xh - f_xh_minus) / (2*h)

print("中心差分の展開結果:")
sympy.pprint(central_diff)

結果を見ると、

$$ \frac{d}{dx} f(x) + \frac{h^2}{6} \frac{d^3}{dx^3} f(x) + O(h^3) $$

となっている。第1項が求めたい微分値であり、第2項以降が誤差である。
誤差の主要項が $h^2$ に比例しているため、これは「2次精度の近似」であることがわかる。

## 3. 高次スキームの自動導出

より高精度な近似や、高階微分（$f''(x)$ など）の近似式が必要な場合、手計算で係数を決めるのは大変である。
ここでは、「未定係数法」を用いて、任意のステンシルに対する差分公式を自動的に導出する手法を紹介する。

例として、5点ステンシル $\{x-2h, x-h, x, x+h, x+2h\}$ を用いて、2階微分 $f''(x)$ を近似する公式を導く。

$$ c_{-2}f(x-2h) + c_{-1}f(x-h) + c_0f(x) + c_1f(x+h) + c_2f(x+2h) \approx f''(x) $$

In [ ]:
# 係数の定義
c_m2, c_m1, c_0, c_1, c_2 = symbols('c_{-2} c_{-1} c_0 c_1 c_2')
coeffs = [c_m2, c_m1, c_0, c_1, c_2]
points = [-2, -1, 0, 1, 2]

# 各点でのテイラー展開（十分高い次数までとる）
order = 6
taylors = [f.subs(x, x + p*h).series(h, 0, order) for p in points]

# 線形結合を作る
linear_comb = sum(c * t for c, t in zip(coeffs, taylors))

# hの各次数の係数を取り出す
# h^0 (定数項) -> 0
# h^1 (1階微分) -> 0
# h^2 (2階微分) -> 1 (これを作りたい)
# h^3 ... -> 0
# h^4 ... -> 0

equations = []
for i in range(5): # 未知数が5個なので式も5個必要
    term = linear_comb.coeff(h, i)
    # term は c_i * f^(i)(x) / i! の形をしているので、係数部分のみ抽出したいが、
    # 単純に f^(i)(x) の係数を比較すればよい。
    # ここでは h^i の係数全体が、ターゲット（0 または 1/2! f''(x)）と一致する条件を課す。
    
    target = 0
    if i == 2:
        # f''(x) の項を作りたい。
        # linear_comb の h^2 の項は (sum c_i * p^2/2) * h^2 * f''(x)
        # これが 1 * f''(x) になってほしいわけではない。
        # 最終的に全体を h^2 で割ったときに f''(x) になってほしい。
        # つまり、linear_comb 自体は h^2 * f''(x) になってほしい。
        # よって h^2 の係数は f''(x) である。
        target = f.diff(x, 2)
    
    equations.append(sympy.Eq(term, target))

print("連立方程式:")
for eq in equations:
    sympy.pprint(eq)

この連立方程式を解いて係数を決定する。

In [ ]:
sol = solve(equations, coeffs)

print("係数の解:")
sympy.pprint(sol)

得られた係数を用いて、差分公式を構成する。全体を $h^2$ で割ることを忘れてはならない（2階微分の近似なので）。

$$ f''(x) \approx \frac{1}{h^2} \sum c_i f(x+ih) $$

In [ ]:
diff_formula = sum(sol[c] * f.subs(x, x + p*h) for c, p in zip(coeffs, points)) / h**2

print("導出された5点差分公式:")
sympy.pprint(diff_formula)

結果は以下のようになった。

$$ f''(x) \approx \frac{-f(x-2h) + 16f(x-h) - 30f(x) + 16f(x+h) - f(x+2h)}{12h^2} $$

これは教科書などに見られる「4次精度中心差分公式」と一致する。

### 3.1 誤差解析

この公式の精度を検証するために、再びテイラー展開を代入して誤差項を確認する。

In [ ]:
# 導出した式をテイラー展開
error_analysis = diff_formula.series(h, 0, 6)

print("誤差解析結果:")
sympy.pprint(error_analysis)

結果は $f''(x) + O(h^4)$ となり、確かに4次精度（誤差が $h^4$ のオーダー）であることが確認できた。

## 結論

この記事では、SymPyを用いて数値解析における差分公式の導出を行った。

テイラー展開を記号的に扱うことで、任意のステンシル配置に対する差分係数を自動的に決定し、その打ち切り誤差まで厳密に評価できることを示した。この手法は、複雑な境界条件や非等間隔グリッドにおける差分スキームを設計する際に極めて有用である。

### 参考文献
- [SymPy Documentation: Series](https://docs.sympy.org/latest/modules/series/index.html)